In [ ]:
# ============================================================
# CELL 1 — Mount Google Drive & Set Seeds
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os, random, json, pickle
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

BASE_PATH         = "/content/drive/MyDrive/Emotion_Story_Project1"
DATA_PATH         = os.path.join(BASE_PATH, "data")
MODEL_PATH        = os.path.join(BASE_PATH, "models")
FACIAL_MODEL_PATH = os.path.join(MODEL_PATH, "facial")
AUDIO_MODEL_PATH  = os.path.join(MODEL_PATH, "audio")
FUSION_MODEL_PATH = os.path.join(MODEL_PATH, "fusion")
OUTPUT_PATH       = os.path.join(BASE_PATH, "outputs")
STORY_OUTPUT_PATH = os.path.join(OUTPUT_PATH, "stories")
METRIC_PATH       = os.path.join(OUTPUT_PATH, "metrics")

for folder in [STORY_OUTPUT_PATH, METRIC_PATH]:
    os.makedirs(folder, exist_ok=True)

print("✅ Seeds configured and directories verified.")

Mounted at /content/drive
✅ Seeds configured and directories verified.


In [ ]:
# ============================================================
# CELL 2 — Install Required Libraries
# ============================================================

# Installs Gemini API, Edge-TTS, and MoviePy dependencies
!apt-get update -y -q
!apt-get install imagemagick -y -q
!sed -i 's/rights="none" pattern="@\*"/rights="read|write" pattern="@\*"/g' /etc/ImageMagick-6/policy.xml

!pip install -q \
    google-generativeai \
    edge-tts \
    moviepy==1.0.3 \
    librosa==0.10.1 \
    gtts \
    requests \
    pillow

import os
os.environ["IMAGEMAGICK_BINARY"] = "/usr/bin/convert"
print("✅ Setup installation complete.")

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,705 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.3 MB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,607 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted am

In [ ]:
# ============================================================
# CELL 3 — Core Imports
# ============================================================

import cv2, time, librosa, warnings, traceback, re, glob, requests, urllib.parse, shutil, gc
from datetime import datetime
from collections import deque
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from PIL import Image
from torchvision import transforms
import google.generativeai as genai
from gtts import gTTS
import edge_tts
from moviepy.editor import ImageClip, concatenate_videoclips, AudioFileClip, TextClip, CompositeVideoClip
from IPython.display import Video, Audio, display, HTML, Javascript
from google.colab import files, output
from base64 import b64decode

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Libraries imported. Device:", device)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate   

✅ Libraries imported. Device: cuda


In [ ]:
# ============================================================
# CELL 4 — Global Configuration Parameters
# ============================================================

# ── Audio DSP ───────────────────────────────────────────────
SR           = 22050
N_MELS       = 128
FIXED_LENGTH = 3
SAMPLES      = SR * FIXED_LENGTH
EPSILON      = 1e-6

# ── Emotion Mapping ─────────────────────────────────────────
COMMON_EMOTIONS = ["Happy", "Neutral", "Sad"]
NUM_CLASSES     = len(COMMON_EMOTIONS)

# ── Temporal Stabilisation & Context ─────────────────────────
BUFFER_SIZE        = 15
EMA_ALPHA          = 0.35
TRANSITION_PENALTY = 0.12
CONTEXT_WINDOW     = 5
CONTEXT_WEIGHT     = 0.15

# ── Reliability & Calibration ───────────────────────────────
MIN_RELIABILITY    = 0.15
CONFLICT_THRESHOLD = 0.60
FACE_TEMPERATURE   = 1.2
AUDIO_TEMPERATURE  = 1.0
UNCERTAINTY_THRESHOLD = 1.10

print("✅ Global configuration loaded successfully.")

✅ Global configuration loaded successfully.


In [ ]:
# ============================================================
# CELL 5 — AudioCNN & Classifier Block definitions
# ============================================================

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2   = nn.BatchNorm2d(channels)
        self.se    = SEBlock(channels)
    def forward(self, x):
        identity = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        return F.relu(out + identity)

class AudioCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.conv1  = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1    = nn.BatchNorm2d(32)
        self.res1   = ResidualBlock(32)
        self.pool1  = nn.MaxPool2d(2)
        self.drop1  = nn.Dropout(0.2)
        self.conv2  = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2    = nn.BatchNorm2d(64)
        self.res2   = ResidualBlock(64)
        self.pool2  = nn.MaxPool2d(2)
        self.drop2  = nn.Dropout(0.3)
        self.conv3  = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3    = nn.BatchNorm2d(128)
        self.pool3  = nn.MaxPool2d(2)
        self.drop3  = nn.Dropout(0.3)
        self.gap    = nn.AdaptiveAvgPool2d(1)
        self.fc1    = nn.Linear(128, 64)
        self.drop_fc = nn.Dropout(0.4)
        self.fc2    = nn.Linear(64, num_classes)
    def forward(self, x):
        x = self.drop1(self.pool1(self.res1(F.relu(self.bn1(self.conv1(x))))))
        x = self.drop2(self.pool2(self.res2(F.relu(self.bn2(self.conv2(x))))))
        x = self.drop3(self.pool3(F.relu(self.bn3(self.conv3(x)))))
        x = self.gap(x).view(x.size(0), -1)
        x = self.drop_fc(F.relu(self.fc1(x)))
        return self.fc2(x)

print("✅ AudioCNN architecture modules registered.")

✅ AudioCNN architecture modules registered.


In [ ]:
# ============================================================
# CELL 6 — Load Pre-trained FER and SER Models
# ============================================================

def load_checkpoint(model, path):
    ckpt = torch.load(path, map_location=device, weights_only=True)
    if isinstance(ckpt, dict) and list(ckpt.keys())[0].startswith("module."):
        ckpt = {k.replace("module.", ""): v for k, v in ckpt.items()}
    model.load_state_dict(ckpt)
    return model

# ── 1. Load Face Model (EfficientNet-B2) ──
face_model_path = os.path.join(FACIAL_MODEL_PATH, "best_face_model_stage2_3class.pth")
assert os.path.exists(face_model_path), f"❌ FER model weights not found at: {face_model_path}"

face_model = models.efficientnet_b2(weights=None)
in_features = face_model.classifier[1].in_features
face_model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(in_features, NUM_CLASSES)
)
face_model = load_checkpoint(face_model, face_model_path).to(device).eval()
print("✅ Pretrained FER model loaded successfully.")

# ── 2. Load Audio Model (Ensemble or Single Seed Fallback) ──
ensemble_bundle_path = os.path.join(AUDIO_MODEL_PATH, "ensemble_audio_model.pth")
single_model_path    = os.path.join(AUDIO_MODEL_PATH, "final_best_audio_model.pth")
audio_models = []

if os.path.exists(ensemble_bundle_path):
    state_dicts = torch.load(ensemble_bundle_path, map_location=device, weights_only=True)
    for sd in state_dicts:
        m = AudioCNN(num_classes=NUM_CLASSES)
        m.load_state_dict(sd)
        m.to(device).eval()
        audio_models.append(m)
    USE_ENSEMBLE = True
    print(f"✅ SER 3-model ensemble loaded successfully.")
else:
    assert os.path.exists(single_model_path), f"❌ No SER models found. Checked single path: {single_model_path}"
    audio_model = AudioCNN(num_classes=NUM_CLASSES)
    audio_model = load_checkpoint(audio_model, single_model_path).to(device).eval()
    audio_models = [audio_model]
    USE_ENSEMBLE = False
    print("⚠️ Ensemble bundle not found — loaded single SER model.")

✅ Pretrained FER model loaded successfully.
✅ SER 3-model ensemble loaded successfully.


In [ ]:
# ============================================================
# CELL 7 — Face Preprocessing Pipeline
# ============================================================

face_detector = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
assert not face_detector.empty(), "❌ Haar Cascade XML failed to load."

face_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def compute_face_quality(gray_face):
    blur = cv2.Laplacian(gray_face, cv2.CV_64F).var()
    brightness = np.mean(gray_face)
    blur_norm = min(blur / 1000.0, 1.0)
    bright_norm = 1.0 - abs(brightness - 127) / 127.0
    return float(np.clip(0.7 * blur_norm + 0.3 * bright_norm, 0.0, 1.0))

def preprocess_face(image_source):
    """
    Accepts file path (str) or BGR frame (np.ndarray).
    Returns: (tensor_batch | None, quality_score, pil_face | None)
    """
    if isinstance(image_source, str):
        image = cv2.imread(image_source)
        if image is None:
            raise ValueError(f"Could not load image: {image_source}")
    else:
        image = image_source.copy()

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))

    if len(faces) == 0:
        return None, 0.0, None

    # Extract largest face box
    x, y, w, h = sorted(faces, key=lambda f: f[2] * f[3], reverse=True)[0]
    face_crop = image[y:y + h, x:x + w]
    gray_crop = gray[y:y + h, x:x + w]
    quality   = compute_face_quality(gray_crop)

    face_rgb = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
    pil_face = Image.fromarray(face_rgb)
    tensor   = face_transform(pil_face)

    return tensor.unsqueeze(0).to(device), quality, pil_face

print("✅ Face preprocessing pipeline ready.")

✅ Face preprocessing pipeline ready.


In [ ]:
# ============================================================
# CELL 8 — Audio Feature Extraction Pipeline
# ============================================================

def generate_features(audio_source):
    """
    Accepts file path (str) or raw 1D array.
    Extracts channels: Log-mel + delta + delta-delta -> normalized stack (3, 128, 130).
    """
    try:
        if isinstance(audio_source, str):
            y, sr = librosa.load(audio_source, sr=SR)
        else:
            y, sr = audio_source, SR

        # Silence trimming
        y, _ = librosa.effects.trim(y, top_db=20)
        # Amplitude Normalization
        y = librosa.util.normalize(y)

        # Padding or cropping to 3 seconds
        if len(y) < SAMPLES:
            y = np.pad(y, (0, SAMPLES - len(y)))
        else:
            y = y[:SAMPLES]

        # Log-mel Spectrogram
        mel = librosa.feature.melspectrogram(
            y=y, sr=sr, n_fft=2048, hop_length=512, win_length=2048, n_mels=N_MELS
        )
        mel_db = librosa.power_to_db(mel, ref=np.max)
        delta  = librosa.feature.delta(mel_db)
        delta2 = librosa.feature.delta(mel_db, order=2)

        # Stack feature channels
        stacked = np.stack([mel_db, delta, delta2], axis=0)

        # Channel-wise standardisation
        for i in range(3):
            stacked[i] = (stacked[i] - stacked[i].mean()) / (stacked[i].std() + EPSILON)

        return stacked.astype(np.float32)
    except Exception as e:
        print(f"❌ Audio extraction error: {e}")
        return None

print("✅ Audio feature extraction ready.")

✅ Audio feature extraction ready.


In [ ]:
# ============================================================
# CELL 9 — Calibration and Entropy Helpers
# ============================================================

def temperature_scale(probs_vector, temperature):
    """
    Applies temperature scaling to soften probability distribution.
    """
    log_p = np.log(np.clip(probs_vector, 1e-9, 1.0))
    scaled = log_p / temperature
    shifted = scaled - np.max(scaled)
    exp_p = np.exp(shifted)
    return exp_p / exp_p.sum()

def prediction_entropy(probs):
    """
    Computes prediction uncertainty via Shannon Entropy (in bits).
    """
    probs = np.clip(probs, 1e-9, 1.0)
    return float(-np.sum(probs * np.log2(probs)))

print("✅ Calibration utilities loaded.")

✅ Calibration utilities loaded.


In [ ]:
# ============================================================
# CELL 10 — Unimodal Prediction Functions (TTA for Face)
# ============================================================

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

TTA_TRANSFORMS = [
    transforms.Compose([transforms.Resize((224, 224)), transforms.Grayscale(3), transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]),
    transforms.Compose([transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(p=1.0), transforms.Grayscale(3), transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]),
    transforms.Compose([transforms.Resize((232, 232)), transforms.CenterCrop(224), transforms.Grayscale(3), transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]),
    transforms.Compose([transforms.Resize((224, 224)), transforms.Grayscale(3), transforms.ColorJitter(brightness=0.1, contrast=0.1), transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]),
]

def predict_face_emotion_tta(image_source):
    try:
        _, quality, pil_face = preprocess_face(image_source)
        if pil_face is None:
            return {"emotion": "Neutral", "probabilities": np.array([0.33, 0.34, 0.33]),
                    "confidence": 0.0, "quality": 0.0, "reliability": 0.0, "entropy": 1.585, "valid": False}

        # Run 4-transform Test Time Augmentation (TTA)
        tta_probs = []
        with torch.inference_mode():
            for tfm in TTA_TRANSFORMS:
                t = tfm(pil_face).unsqueeze(0).to(device)
                out = face_model(t)
                prob = torch.softmax(out, dim=1).cpu().numpy()[0]
                tta_probs.append(prob)

        face_probs = np.mean(tta_probs, axis=0)
        # Temperature calibration scaling
        face_probs = temperature_scale(face_probs, FACE_TEMPERATURE)

        pred_idx = int(np.argmax(face_probs))
        confidence = float(np.max(face_probs))
        reliability = 0.7 * confidence + 0.3 * quality
        entropy = prediction_entropy(face_probs)

        return {
            "emotion": COMMON_EMOTIONS[pred_idx],
            "probabilities": face_probs,
            "confidence": confidence,
            "quality": quality,
            "reliability": reliability,
            "entropy": entropy,
            "valid": True
        }
    except Exception as e:
        print(f"❌ Face prediction error: {e}")
        return {"emotion": "Neutral", "probabilities": np.array([0.33, 0.34, 0.33]), "confidence": 0.0, "quality": 0.0, "reliability": 0.0, "entropy": 1.585, "valid": False}

def predict_audio_emotion(audio_source):
    try:
        if isinstance(audio_source, str):
            y_raw, _ = librosa.load(audio_source, sr=SR)
        else:
            y_raw = audio_source

        rms = float(np.sqrt(np.mean(y_raw ** 2))) if len(y_raw) > 0 else 0.0
        quality = min(rms / 0.1, 1.0)

        feats = generate_features(audio_source)
        if feats is None:
            raise ValueError("Acoustic features failed to extract.")

        tensor = torch.tensor(feats).unsqueeze(0).to(device)

        with torch.inference_mode():
            ensemble_probs = None
            for m in audio_models:
                p = torch.softmax(m(tensor), dim=1)
                ensemble_probs = p if ensemble_probs is None else ensemble_probs + p
            probs = (ensemble_probs / len(audio_models)).cpu().numpy()[0]

        # ── Temperature calibration ──────────────────────────
        # FIXED: Variable referenced correctly as probs (resolving Phase 3 bug)
        audio_probs = temperature_scale(probs, AUDIO_TEMPERATURE)

        pred_idx = int(np.argmax(audio_probs))
        confidence = float(np.max(audio_probs))
        reliability = 0.6 * confidence + 0.4 * quality
        entropy = prediction_entropy(audio_probs)

        return {
            "emotion": COMMON_EMOTIONS[pred_idx],
            "probabilities": audio_probs,
            "confidence": confidence,
            "quality": quality,
            "reliability": reliability,
            "entropy": entropy,
            "valid": True
        }
    except Exception as e:
        print(f"❌ Audio prediction error: {e}")
        return {"emotion": "Neutral", "probabilities": np.array([0.33, 0.34, 0.33]), "confidence": 0.0, "quality": 0.0, "reliability": 0.0, "entropy": 1.585, "valid": False}

print("✅ Unimodal prediction functions registered.")

✅ Unimodal prediction functions registered.


In [ ]:
# ============================================================
# CELL 11 — Context Awareness Module
# ============================================================

class ContextAwareness:
    def __init__(self, window=CONTEXT_WINDOW, num_classes=NUM_CLASSES):
        self.window = window
        self.num_classes = num_classes
        self.history = deque(maxlen=window)

    def update(self, emotion):
        idx = COMMON_EMOTIONS.index(emotion)
        self.history.append(idx)

    def get_prior(self):
        if len(self.history) == 0:
            return np.ones(self.num_classes) / self.num_classes

        counts = np.zeros(self.num_classes)
        for rank, idx in enumerate(self.history):
            weight = (rank + 1) / len(self.history)
            counts[idx] += weight

        return counts / counts.sum()

    def dominant_emotion(self):
        if len(self.history) == 0:
            return "Unknown"
        counts = np.bincount(list(self.history), minlength=self.num_classes)
        return COMMON_EMOTIONS[int(np.argmax(counts))]

    def reset(self):
        self.history.clear()

session_context = ContextAwareness()
print("✅ Context tracker initialized.")

✅ Context tracker initialized.


In [ ]:
# ============================================================
# CELL 12 — Learned MLP Fusion Model
# ============================================================

FUSION_INPUT_DIM = NUM_CLASSES * 3 + 2

class FusionMLP(nn.Module):
    def __init__(self, input_dim=FUSION_INPUT_DIM, hidden=64, num_classes=NUM_CLASSES):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden // 2, num_classes)
        )
    def forward(self, x):
        return self.net(x)

# Load model weights
fusion_bundle_path = os.path.join(FUSION_MODEL_PATH, "fusion_bundle.pt")
assert os.path.exists(fusion_bundle_path), f"❌ Fusion weights not found at: {fusion_bundle_path}"

bundle = torch.load(fusion_bundle_path, map_location=device, weights_only=True)
fusion_model = FusionMLP(input_dim=bundle["fusion_input_dim"], hidden=bundle["fusion_hidden"], num_classes=bundle["num_classes"])
fusion_model.load_state_dict(bundle["fusion_mlp_state"])
fusion_model = fusion_model.to(device).eval()

print("✅ Learned FusionMLP loaded and configured.")

✅ Learned FusionMLP loaded and configured.


In [ ]:
# ============================================================
# CELL 13 — Context-Aware Learned Fusion & Stabilizer
# ============================================================

def fuse_emotions_learned(face_result, audio_result, context=None):
    face_valid  = face_result["valid"]
    audio_valid = audio_result["valid"]

    if not face_valid and not audio_valid:
        return {"emotion": "Neutral", "probabilities": np.array([0.33, 0.34, 0.33]),
                "confidence": 0.0, "face_weight": 0.0, "audio_weight": 0.0,
                "conflict": False, "uncertain": True, "context_emotion": "Unknown"}

    if face_valid and not audio_valid:
        probs = face_result["probabilities"]
        return {"emotion": face_result["emotion"], "probabilities": probs,
                "confidence": face_result["confidence"], "face_weight": 1.0, "audio_weight": 0.0,
                "conflict": False, "uncertain": prediction_entropy(probs) > UNCERTAINTY_THRESHOLD, "context_emotion": "N/A"}

    if audio_valid and not face_valid:
        probs = audio_result["probabilities"]
        return {"emotion": audio_result["emotion"], "probabilities": probs,
                "confidence": audio_result["confidence"], "face_weight": 0.0, "audio_weight": 1.0,
                "conflict": False, "uncertain": prediction_entropy(probs) > UNCERTAINTY_THRESHOLD, "context_emotion": "N/A"}

    # Both valid: compute MLP inference features
    ctx_prior = context.get_prior() if context else np.ones(NUM_CLASSES) / NUM_CLASSES

    feat_vec = np.concatenate([
        face_result["probabilities"],
        audio_result["probabilities"],
        ctx_prior,
        [max(face_result["reliability"], MIN_RELIABILITY),
         max(audio_result["reliability"], MIN_RELIABILITY)]
    ]).astype(np.float32)

    tensor = torch.tensor(feat_vec).unsqueeze(0).to(device)
    with torch.inference_mode():
        logits = fusion_model(tensor)
        fused_probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

    # Blend context history prior
    fused_probs = (1 - CONTEXT_WEIGHT) * fused_probs + CONTEXT_WEIGHT * ctx_prior
    fused_probs = fused_probs / fused_probs.sum()

    final_idx = int(np.argmax(fused_probs))
    final_emotion = COMMON_EMOTIONS[final_idx]
    confidence = float(np.max(fused_probs))
    entropy = prediction_entropy(fused_probs)
    uncertain = entropy > UNCERTAINTY_THRESHOLD

    conflict = (
        face_result["emotion"] != audio_result["emotion"]
        and face_result["confidence"] > CONFLICT_THRESHOLD
        and audio_result["confidence"] > CONFLICT_THRESHOLD
    )

    fr = max(face_result["reliability"], MIN_RELIABILITY)
    ar = max(audio_result["reliability"], MIN_RELIABILITY)
    total = fr + ar

    return {
        "emotion": final_emotion,
        "probabilities": fused_probs,
        "confidence": confidence,
        "face_weight": fr / total,
        "audio_weight": ar / total,
        "conflict": conflict,
        "uncertain": uncertain,
        "context_emotion": context.dominant_emotion() if context else "N/A"
    }

class TemporalStabilizer:
    def __init__(self, alpha=EMA_ALPHA, penalty=TRANSITION_PENALTY, buffer_size=BUFFER_SIZE):
        self.alpha = alpha
        self.penalty = penalty
        self.ema_probs = None
        self.prev_emotion = None
        self.history = deque(maxlen=buffer_size)

    def update(self, fusion_result):
        probs = fusion_result["probabilities"]
        curr_emo = fusion_result["emotion"]
        self.history.append(probs)

        if self.ema_probs is None:
            self.ema_probs = probs.copy()
        else:
            self.ema_probs = self.alpha * probs + (1 - self.alpha) * self.ema_probs

        smoothed = self.ema_probs.copy()
        if self.prev_emotion is not None and curr_emo != self.prev_emotion:
            curr_idx = COMMON_EMOTIONS.index(curr_emo)
            prev_idx = COMMON_EMOTIONS.index(self.prev_emotion)
            smoothed[curr_idx] -= self.penalty
            smoothed[prev_idx] += self.penalty

        smoothed = np.clip(smoothed, 1e-6, None)
        smoothed /= smoothed.sum()
        stable_idx = int(np.argmax(smoothed))
        stable_emo = COMMON_EMOTIONS[stable_idx]
        self.prev_emotion = stable_emo

        return {
            "emotion": stable_emo,
            "probabilities": smoothed,
            "confidence": float(np.max(smoothed))
        }

    def reset(self):
        self.ema_probs = None
        self.prev_emotion = None
        self.history.clear()

stabilizer = TemporalStabilizer()
print("✅ Multimodal Decision Fusion ready.")

✅ Multimodal Decision Fusion ready.


In [ ]:
# ============================================================
# CELL 14 — HTML5/JS Live Webcam Image Capture Widget
# ============================================================

def take_webcam_photo(filename='webcam_capture.jpg', quality=0.95):
    js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = '📸 Capture Snapshot';
      capture.style.cssText = 'padding:10px 20px; font-size:14px; background:#4CAF50; color:white; border:none; border-radius:4px; cursor:pointer; margin-bottom:10px;';

      const video = document.createElement('video');
      video.style.display = 'block';
      video.style.borderRadius = '8px';
      video.style.marginBottom = '10px';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      div.appendChild(capture);
      video.srcObject = stream;
      await video.play();

      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
    display(js)
    try:
        data = output.eval_js('takePhoto({})'.format(quality))
        binary = b64decode(data.split(',')[1])
        with open(filename, 'wb') as f:
            f.write(binary)
        print(f"✅ Webcam snapshot saved: {filename}")
        return filename
    except Exception as e:
        print(f"⚠️ Webcam error (permission denied or no device): {e}")
        return None

In [ ]:
# ============================================================
# CELL 15 — HTML5/JS Live Audio Voice Recording Widget
# ============================================================

def record_audio_clip(filename='microphone_capture.wav', duration=3):
    js = Javascript('''
    async function recordAudio(durationMs) {
      const div = document.createElement('div');
      const label = document.createElement('p');
      label.textContent = '🔴 Ready to record...';
      label.style.fontFamily = 'sans-serif';
      label.style.fontWeight = 'bold';

      const btn = document.createElement('button');
      btn.textContent = '🎤 Start Recording (3s)';
      btn.style.cssText = 'padding:10px 20px; font-size:14px; background:#F44336; color:white; border:none; border-radius:4px; cursor:pointer;';

      div.appendChild(label);
      div.appendChild(btn);
      document.body.appendChild(div);

      await new Promise((resolve) => btn.onclick = resolve);
      label.textContent = '🎙️ Recording - Speak now...';
      btn.disabled = true;

      const stream = await navigator.mediaDevices.getUserMedia({audio: true});
      const mediaRecorder = new MediaRecorder(stream);
      const chunks = [];

      mediaRecorder.ondataavailable = (e) => chunks.push(e.data);
      mediaRecorder.start();

      await new Promise((resolve) => setTimeout(resolve, durationMs));
      mediaRecorder.stop();
      stream.getAudioTracks()[0].stop();
      label.textContent = '⌛ Processing audio...';

      await new Promise((resolve) => mediaRecorder.onstop = resolve);
      const blob = new Blob(chunks, {type: 'audio/webm'});
      const reader = new FileReader();

      return new Promise((resolve) => {
        reader.onloadend = () => {
          div.remove();
          resolve(reader.result);
        };
        reader.readAsDataURL(blob);
      });
    }
    ''')
    display(js)
    try:
        data = output.eval_js('recordAudio({})'.format(duration * 1000))
        binary = b64decode(data.split(',')[1])

        temp_webm = "temp_record.webm"
        with open(temp_webm, 'wb') as f:
            f.write(binary)

        clip = AudioFileClip(temp_webm)
        clip.write_audiofile(filename, fps=SR, nbytes=4, codec='pcm_s16le', verbose=False, logger=None)
        clip.close()
        os.remove(temp_webm)

        print(f"✅ Microphone clip saved: {filename}")
        return filename
    except Exception as e:
        print(f"⚠️ Microphone error (permission denied or no device): {e}")
        return None

In [ ]:
# ============================================================
# CELL 16 — Unified User Input Interface
# ============================================================

print("=== CHOOSE FACE INPUT MODE ===")
print("1. Upload Image File")
print("2. Live Webcam Snapshot")
print("3. Skip Face Modality")
face_choice = input("Enter choice (1-3): ").strip()

face_file = None
if face_choice == '1':
    print("\n📤 Click below to upload a portrait image (JPG / PNG):")
    uploaded = files.upload()
    if uploaded:
        face_file = list(uploaded.keys())[0]
elif face_choice == '2':
    face_file = take_webcam_photo('live_face.jpg')
else:
    print("⏭️ Skipping face input.")

print("\n=== CHOOSE AUDIO INPUT MODE ===")
print("1. Upload WAV File")
print("2. Record Live Voice clip")
print("3. Skip Audio Modality")
audio_choice = input("Enter choice (1-3): ").strip()

audio_file = None
if audio_choice == '1':
    print("\n📤 Click below to upload a voice file (WAV):")
    uploaded = files.upload()
    if uploaded:
        audio_file = list(uploaded.keys())[0]
elif audio_choice == '2':
    audio_file = record_audio_clip('live_audio.wav')
else:
    print("⏭️ Skipping audio input.")

print("\n✅ Media capture workflow complete.")

=== CHOOSE FACE INPUT MODE ===
1. Upload Image File
2. Live Webcam Snapshot
3. Skip Face Modality
Enter choice (1-3): 1

📤 Click below to upload a portrait image (JPG / PNG):


Saving happy face7.png to happy face7.png

=== CHOOSE AUDIO INPUT MODE ===
1. Upload WAV File
2. Record Live Voice clip
3. Skip Audio Modality
Enter choice (1-3): 1

📤 Click below to upload a voice file (WAV):


Saving happy_audio.wav to happy_audio.wav

✅ Media capture workflow complete.


In [ ]:
# ============================================================
# CELL 17 — Predict Multimodal Emotion
# ============================================================

face_res = predict_face_emotion_tta(face_file) if face_file else {"valid": False}
audio_res = predict_audio_emotion(audio_file) if audio_file else {"valid": False}

# Compute context-aware fusion
fusion_res = fuse_emotions_learned(face_res, audio_res, session_context)
stable_res = stabilizer.update(fusion_res)
session_context.update(stable_res["emotion"])

DETECTED_EMOTION = stable_res["emotion"]
stable_uncertain = prediction_entropy(stable_res['probabilities']) > UNCERTAINTY_THRESHOLD

print("\n" + "=" * 50)
print("🎯 FINAL EMOTION RESULTS")
print("=" * 50)
print(f"  Face Modality  : {face_res.get('emotion', 'Skipped')} (Confidence: {face_res.get('confidence', 0.0):.2%})")
print(f"  Audio Modality : {audio_res.get('emotion', 'Skipped')} (Confidence: {audio_res.get('confidence', 0.0):.2%})")
print(f"  Fused Predict  : {fusion_res['emotion']} (Conflict Flag: {fusion_res['conflict']})")
print(f"  Stable Outcome : {DETECTED_EMOTION} (Uncertain Flag: {stable_uncertain})")
print("=" * 50)


🎯 FINAL EMOTION RESULTS
  Face Modality  : Happy (Confidence: 99.88%)
  Audio Modality : Happy (Confidence: 54.94%)
  Fused Predict  : Happy (Conflict Flag: False)
  Stable Outcome : Happy (Uncertain Flag: False)


In [ ]:
# ============================================================
# CELL 18 — Story Text & Visual Scene Prompts Generation (Gemini)
# ============================================================

GEMINI_API_KEY = "" # ⚠️ Insert your Gemini API Key here

if not GEMINI_API_KEY:
    print("⚠️ Gemini API Key missing! Fallback to default dummy story.")
    story_content = {
        "story": "Once upon a time, a little puppy named Bruno was resting under the warm sun. He looked up at the sky and saw beautiful butterflies flying in circles. Aarav walked up to Bruno and hugged him tightly. They spent the evening playing together in the park, feeling extremely happy.",
        "scenes": [
            "Bruno the puppy resting under a warm sun, cartoon drawing style",
            "Butterflies flying in circles in a bright sky, colorful, illustration",
            "Young boy Aarav hugging a happy mud-covered puppy, emotional lighting",
            "A boy and puppy playing in a green garden, detailed, digital art"
        ]
    }
else:
    genai.configure(api_key=GEMINI_API_KEY)
    model = genai.GenerativeModel('gemini-flash-latest')

    # Craft prompt tailored to emotion
    prompt = f"""Write a soothing bedtime story for a child whose detected emotion is {DETECTED_EMOTION}.
    - If emotion is Sad: Make it a deeply comforting, secure, warm story about reassurance, friendship, and sleeping peacefully.
    - If emotion is Happy: Make it a fun, lighthearted, gentle adventure about curiosity and starry skies.
    - If emotion is Neutral: Make it a relaxing, rhythmic tale focusing on peaceful nature sounds and falling asleep.

    Write exactly 4 paragraphs (around 300-400 words total).
    Additionally, provide exactly 4 short illustration scene prompts (one for each paragraph) that represent the visual progression.
    Return your response strictly in the following JSON format:
    {{
      "story": "...full text of the story...",
      "scenes": ["scene prompt 1", "scene prompt 2", "scene prompt 3", "scene prompt 4"]
    }}
    """

    try:
        response = model.generate_content(
            prompt,
            generation_config={"response_mime_type": "application/json"}
        )
        story_content = json.loads(response.text)
        print("✅ Story and scene prompts generated via Gemini.")
    except Exception as e:
        print(f"❌ Gemini API error: {e}. Falling back to default story.")
        story_content = {
            "story": "A little boy Aarav and his puppy Bruno loved exploring the hills behind their home. Today, they saw stars shining in the night sky. Aarav smiled, feeling safe and sleepy in his bed.",
            "scenes": [
                "A boy and puppy looking at hills, cartoon style",
                "Starry night sky above green hills, warm lighting",
                "A child sleeping peacefully in a cozy room, illustration"
            ]
        }

print("\nGenerated Story Text:")
print(story_content["story"])

✅ Story and scene prompts generated via Gemini.

Generated Story Text:
Pip the squirrel wasn't sleepy yet; his tail was still full of happy little wiggles. From his cozy treehouse branch, he looked up at the night sky and noticed that the stars weren't just shining—they were playing a gentle game of connect-the-dots. With a soft giggle, Pip hopped onto a fluffy, floating cloud that had drifted close to his balcony, ready for a midnight safari through the twinkling cosmos.

The cloud carried Pip higher, gliding past a crescent moon that looked like a giant slice of golden melon. Along the way, he met Oliver, a sleepy owl wearing a tiny nightcap, who was busy dusting the stars with a soft feather wand to make them sparklier. Together, they chased a friendly shooting star that looped and spun through the deep blue sky, leaving a trail of shimmering, lavender stardust behind it.

"Where do the stars go when they blink?" Pip wondered aloud. Oliver smiled and pointed to a cosmic playground w

In [ ]:
# ============================================================
# CELL 19 — Story Voice Narration TTS
# ============================================================

ELEVENLABS_API_KEY = "" # ⚠️ Optional ElevenLabs key
narration_file = "story_narration.mp3"

# Helper for Edge-TTS
def get_edge_tts_params(emotion):
    if emotion.lower() == "happy":
        return "en-US-GuyNeural", "+12%", "+4Hz"
    elif emotion.lower() == "sad":
        return "en-US-JennyNeural", "-18%", "-4Hz"
    else:
        return "en-US-AriaNeural", "0%", "0Hz"

if ELEVENLABS_API_KEY:
    print("🎙️ Generating narration via ElevenLabs...")
    voice_id = "EXAVITQu4vr4xnSDxMaL"
    url = f"https://api.elevenlabs.io/v1/text-to-speech/{voice_id}"
    headers = {"xi-api-key": ELEVENLABS_API_KEY, "Content-Type": "application/json"}
    data = {
        "text": story_content["story"],
        "model_id": "eleven_multilingual_v2",
        "voice_settings": {"stability": 0.35, "similarity_boost": 0.85, "style": 0.6, "use_speaker_boost": True}
    }
    response = requests.post(url, json=data, headers=headers)
    if response.status_code == 200:
        with open(narration_file, 'wb') as f:
            f.write(response.content)
        print("✅ ElevenLabs audio file generated.")
    else:
        print(f"⚠️ ElevenLabs error: {response.text}. Falling back to Edge-TTS.")
        ELEVENLABS_API_KEY = None

if not ELEVENLABS_API_KEY:
    print("🎙️ Generating emotion-calibrated narration via Edge-TTS...")
    voice, rate, pitch = get_edge_tts_params(DETECTED_EMOTION)
    communicate = edge_tts.Communicate(
        text=story_content["story"],
        voice=voice,
        rate=rate,
        pitch=pitch
    )
    import asyncio
    loop = asyncio.get_event_loop()
    loop.run_until_complete(communicate.save(narration_file))
    print("✅ Edge-TTS audio file generated.")

display(Audio(narration_file))

🎙️ Generating narration via ElevenLabs...
✅ ElevenLabs audio file generated.


In [ ]:
# ============================================================
# CELL 20 — Visual Scene Image Generation (Pollinations AI)
# ============================================================

# ── Optional API Key ──────────────────────────────────────────
# If you run into 402 (Queue full) or 404 errors, please get a
# free API key from https://enter.pollinations.ai and paste it below.
POLLINATIONS_API_KEY = ""

print("🎨 Fetching illustration frames from Pollinations.ai...")
image_files = []

for i, scene in enumerate(story_content["scenes"]):
    prompt = f"cinematic fairytale cartoon illustration, {scene}, emotional lighting, high resolution, cozy"
    encoded_prompt = urllib.parse.quote(prompt)

    # Route to the correct endpoint based on whether an API key is provided
    if POLLINATIONS_API_KEY.strip():
        url = f"https://gen.pollinations.ai/image/{encoded_prompt}?key={POLLINATIONS_API_KEY.strip()}"
    else:
        url = f"https://image.pollinations.ai/prompt/{encoded_prompt}"

    print(f"  Downloading scene {i+1}: {scene[:50]}...")
    filename = f"story_scene_{i+1}.png"

    max_retries = 3
    timeout_seconds = 30

    for attempt in range(max_retries):
        try:
            # We set standard browser headers to prevent WAF blocks
            headers = {
                "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
            }
            response = requests.get(url, headers=headers, timeout=timeout_seconds)
            if response.status_code == 200 and response.content:
                with open(filename, 'wb') as f:
                    f.write(response.content)
                image_files.append(filename)
                print(f"  ✅ Scene {i+1} saved: {filename}")
                break
            else:
                print(f"    ⚠️ Attempt {attempt+1} failed: Status {response.status_code}")
                if response.status_code in [402, 404] and not POLLINATIONS_API_KEY.strip():
                    print(f"   💡 TIP: Status {response.status_code} is common when running anonymously from shared server/Colab IPs.")
                    print("Get a FREE API key from https://enter.pollinations.ai and set it in POLLINATIONS_API_KEY above to bypass this.")
        except (requests.exceptions.RequestException, requests.exceptions.Timeout) as e:
            print(f"    ⚠️ Attempt {attempt+1} failed with error: {e}")

        # Sleep for a bit before retrying
        if attempt < max_retries - 1:
            time.sleep(3)
    else:
        # All retries failed
        print(f"    ❌ Failed to download scene {i+1} after {max_retries} attempts. Creating placeholder.")
        # Create a placeholder image
        Image.new('RGB', (1280, 720), color = (100, 100, 100)).save(filename)
        image_files.append(filename)

# After the loop, add an extra check to ensure image_files is not empty
if not image_files:
    print("⚠️ No images were generated. Creating a default blank image to proceed.")
    filename = "story_scene_default.png"
    Image.new('RGB', (1280, 720), color = (50, 50, 50)).save(filename)
    image_files.append(filename)

print("✅ All story illustration slides processed.")

🎨 Fetching illustration frames from Pollinations.ai...
  ✅ Scene 1 saved: story_scene_1.png
  ✅ Scene 2 saved: story_scene_2.png
  ✅ Scene 3 saved: story_scene_3.png
  ✅ Scene 4 saved: story_scene_4.png
✅ All story illustration slides processed.


In [ ]:
# ============================================================
# CELL 21 — MoviePy Bedtime Video Assembly
# ============================================================

print("🎬 Compiling final bedtime story video...")

def create_subtitles(text, audio_duration):
    sentences = re.split(r'(?<=[.!?]) +', text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]
    total_chars = sum(len(s) for s in sentences)

    subtitles = []
    current_time = 0.0
    for s in sentences:
        duration = audio_duration * (len(s) / total_chars)
        subtitles.append((current_time, current_time + duration, s))
        current_time += duration
    return subtitles

audio_clip = AudioFileClip(narration_file)
total_duration = audio_clip.duration
slide_duration = total_duration / len(image_files)

clips = []
for img in image_files:
    clip = ImageClip(img).set_duration(slide_duration).resize(height=720)
    clips.append(clip)

base_video = concatenate_videoclips(clips, method="compose")

subs = create_subtitles(story_content["story"], total_duration)
txt_clips = []

for start, end, txt in subs:
    txt_clip = (
        TextClip(
            txt,
            fontsize=26,
            color='white',
            font='DejaVu-Sans-Bold',
            stroke_color='black',
            stroke_width=1.5,
            method='caption',
            size=(800, None)
        )
        .set_position(("center", 600))
        .set_start(start)
        .set_duration(end - start)
    )
    txt_clips.append(txt_clip)

final_video = CompositeVideoClip([base_video] + txt_clips)
final_video = final_video.set_audio(audio_clip)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_video_path = os.path.join(STORY_OUTPUT_PATH, f"personalized_bedtime_story_{timestamp}.mp4")
final_video.write_videofile(
    output_video_path,
    fps=24,
    codec="libx264",
    audio_codec="aac",
    verbose=False,
    logger=None
)

audio_clip.close()
final_video.close()
print(f"✅ Final video ready at: {output_video_path}")

for f in image_files:
    if os.path.exists(f): os.remove(f)

Video(output_video_path, embed=True)

🎬 Compiling final bedtime story video...
✅ Final video ready at: /content/drive/MyDrive/Emotion_Story_Project1/outputs/stories/personalized_bedtime_story_20260606_182748.mp4
